## Day 2 Banking Assignment
Create/use a banking transactions DataFrame with: `transaction_id, customer_id, account_type, transaction_type, amount, city, transaction_date`.

Tasks:
1. Select transaction ID, customer ID and amount.
2. Filter transactions above ₹50,000.
3. Filter only `DEBIT` transactions.
4. Create `transaction_fee` as 1% of amount.
5. Create `net_amount` after fee.
6. Rename `city` to `branch_city`.
7. Drop an unnecessary column.
8. Find distinct account types.
9. Remove duplicate transaction IDs.
10. Sort transactions by amount descending.
11. Group by account type and calculate count, total amount, average amount, min and max.
12. Create `risk_level`: HIGH >= ₹100,000, MEDIUM >= ₹50,000, otherwise LOW.
13. Convert transaction date to DateType and extract year/month/day.
14. Handle null amount/city values appropriately.
15. Create another DataFrame and perform `unionByName`.
16. Create a customer master DataFrame and join it with transactions.
17. Repartition by account type and explain why.
18. Build a final banking pipeline producing clean, enriched transactions.

**Challenge:** Classify the transformations you used as narrow or wide and identify where a shuffle is likely to occur.

In [2]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("BankingAssignment") \
    .getOrCreate()

banking_data = [
    (1, 101, "SAVINGS", "CREDIT", 75000.0, "Bengaluru", "2026-09-01"),
    (2, 102, "CURRENT", "DEBIT", 45000.0, "Hyderabad", "2026-09-02"),
    (3, 103, "SAVINGS", "CREDIT", 120000.0, "Chennai", "2026-09-03"),
    (4, 104, "CURRENT", "DEBIT", 65000.0, "Pune", "2026-09-04"),
    (5, 105, "SAVINGS", "DEBIT", 30000.0, "Mumbai", "2026-09-05"),
    (6, 101, "CURRENT", "CREDIT", 150000.0, "Bengaluru", "2026-09-06"),
    (7, 106, "SAVINGS", "DEBIT", 55000.0, "Delhi", "2026-09-07"),
    (8, 107, "CURRENT", "CREDIT", 90000.0, "Pune", "2026-09-08"),
    (9, 108, "SAVINGS", "DEBIT", 25000.0, "Chennai", "2026-09-09"),
    (10, 109, "CURRENT", "CREDIT", 200000.0, "Hyderabad", "2026-09-10")
]

columns = [
    "transaction_id",
    "customer_id",
    "account_type",
    "transaction_type",
    "amount",
    "city",
    "transaction_date"
]

bank_df = spark.createDataFrame(banking_data, columns)

bank_df.show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|
|             7|        106|     SAVINGS|           DEBIT| 55000.0|    Delhi|      2026-09-07|
|             8|        107|     CURRENT|         

In [3]:
# Question 1
bank_df.select(
    "transaction_id",
    "customer_id",
    "amount"
).show()

+--------------+-----------+--------+
|transaction_id|customer_id|  amount|
+--------------+-----------+--------+
|             1|        101| 75000.0|
|             2|        102| 45000.0|
|             3|        103|120000.0|
|             4|        104| 65000.0|
|             5|        105| 30000.0|
|             6|        101|150000.0|
|             7|        106| 55000.0|
|             8|        107| 90000.0|
|             9|        108| 25000.0|
|            10|        109|200000.0|
+--------------+-----------+--------+



In [ ]:
# Question 2
bank_df.filter(
    col("amount") > 50000
).show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|
|             7|        106|     SAVINGS|           DEBIT| 55000.0|    Delhi|      2026-09-07|
|             8|        107|     CURRENT|          CREDIT| 90000.0|     Pune|      2026-09-08|
|            10|        109|     CURRENT|          CREDIT|200000.0|Hyderabad|      2026-09-10|
+--------------+-----------+------------+---------

In [ ]:
# Question 3
bank_df.filter(
    col("transaction_type") == "DEBIT"
).show()

+--------------+-----------+------------+----------------+-------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type| amount|     city|transaction_date|
+--------------+-----------+------------+----------------+-------+---------+----------------+
|             2|        102|     CURRENT|           DEBIT|45000.0|Hyderabad|      2026-09-02|
|             4|        104|     CURRENT|           DEBIT|65000.0|     Pune|      2026-09-04|
|             5|        105|     SAVINGS|           DEBIT|30000.0|   Mumbai|      2026-09-05|
|             7|        106|     SAVINGS|           DEBIT|55000.0|    Delhi|      2026-09-07|
|             9|        108|     SAVINGS|           DEBIT|25000.0|  Chennai|      2026-09-09|
+--------------+-----------+------------+----------------+-------+---------+----------------+



In [4]:
# Question 4
bank_df.withColumn(
    "transaction_fee",
    col("amount") * 0.01
).show()

+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|transaction_fee|
+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|          750.0|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|          450.0|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|         1200.0|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|          650.0|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|          300.0|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|         1500.0|
|

In [5]:
# Question 5
bank_df.withColumn(
    "transaction_fee",
    col("amount") * 0.01
).withColumn(
    "net_amount",
    col("amount") - col("transaction_fee")
).show()

+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+----------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|transaction_fee|net_amount|
+--------------+-----------+------------+----------------+--------+---------+----------------+---------------+----------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|          750.0|   74250.0|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|          450.0|   44550.0|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|         1200.0|  118800.0|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|          650.0|   64350.0|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|          300.0|   29700.0|
|             6|        

In [6]:
# Question 6
bank_df.withColumnRenamed(
    "city",
    "branch_city"
).show()

+--------------+-----------+------------+----------------+--------+-----------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|branch_city|transaction_date|
+--------------+-----------+------------+----------------+--------+-----------+----------------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|  Bengaluru|      2026-09-01|
|             2|        102|     CURRENT|           DEBIT| 45000.0|  Hyderabad|      2026-09-02|
|             3|        103|     SAVINGS|          CREDIT|120000.0|    Chennai|      2026-09-03|
|             4|        104|     CURRENT|           DEBIT| 65000.0|       Pune|      2026-09-04|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|     Mumbai|      2026-09-05|
|             6|        101|     CURRENT|          CREDIT|150000.0|  Bengaluru|      2026-09-06|
|             7|        106|     SAVINGS|           DEBIT| 55000.0|      Delhi|      2026-09-07|
|             8|        107|  

In [7]:
# Question 7
bank_df.select(
    "account_type"
).distinct().show()

+------------+
|account_type|
+------------+
|     SAVINGS|
|     CURRENT|
+------------+



In [8]:
# Question 8
bank_df.dropDuplicates(
    ["transaction_id"]
).show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|
|             7|        106|     SAVINGS|           DEBIT| 55000.0|    Delhi|      2026-09-07|
|             8|        107|     CURRENT|         

In [9]:
# Question 9
bank_df.orderBy(
    col("amount").desc()
).show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|            10|        109|     CURRENT|          CREDIT|200000.0|Hyderabad|      2026-09-10|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|
|             8|        107|     CURRENT|          CREDIT| 90000.0|     Pune|      2026-09-08|
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|
|             7|        106|     SAVINGS|           DEBIT| 55000.0|    Delhi|      2026-09-07|
|             2|        102|     CURRENT|         

In [10]:
# Question 10
bank_df.groupBy(
    "account_type"
).agg(
    count("transaction_id").alias("transaction_count"),
    sum("amount").alias("total_amount"),
    avg("amount").alias("average_amount"),
    min("amount").alias("minimum_amount"),
    max("amount").alias("maximum_amount")
).show()

+------------+-----------------+------------+--------------+--------------+--------------+
|account_type|transaction_count|total_amount|average_amount|minimum_amount|maximum_amount|
+------------+-----------------+------------+--------------+--------------+--------------+
|     SAVINGS|                5|    305000.0|       61000.0|       25000.0|      120000.0|
|     CURRENT|                5|    550000.0|      110000.0|       45000.0|      200000.0|
+------------+-----------------+------------+--------------+--------------+--------------+



In [11]:
# Question 11
risk_df = bank_df.withColumn(
    "risk_level",
    when(col("amount") >= 100000, "HIGH")
     .when(col("amount") >= 50000, "MEDIUM")
     .otherwise("LOW")
)

risk_df.show()

+--------------+-----------+------------+----------------+--------+---------+----------------+----------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|risk_level|
+--------------+-----------+------------+----------------+--------+---------+----------------+----------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|    MEDIUM|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|       LOW|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|      HIGH|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|    MEDIUM|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|       LOW|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|      HIGH|
|             7|        106|     SAVINGS|     

In [12]:
# Question 12
date_df = bank_df.withColumn(
    "transaction_date",
    to_date("transaction_date")
).withColumn(
    "year",
    year("transaction_date")
).withColumn(
    "month",
    month("transaction_date")
).withColumn(
    "day",
    dayofmonth("transaction_date")
)

date_df.show()

+--------------+-----------+------------+----------------+--------+---------+----------------+----+-----+---+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|year|month|day|
+--------------+-----------+------------+----------------+--------+---------+----------------+----+-----+---+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|2026|    9|  1|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|2026|    9|  2|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|2026|    9|  3|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|2026|    9|  4|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|2026|    9|  5|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|2026|    9|  6|
|         

In [13]:
# Question 13
clean_df = bank_df.fillna({
    "amount": 0,
    "city": "Unknown"
})

clean_df.show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|
|             7|        106|     SAVINGS|           DEBIT| 55000.0|    Delhi|      2026-09-07|
|             8|        107|     CURRENT|         

In [14]:
# Question 14
new_data = [
    (11, 110, "SAVINGS", "CREDIT", 85000.0, "Kolkata", "2026-09-11"),
    (12, 111, "CURRENT", "DEBIT", 40000.0, "Delhi", "2026-09-12")
]

new_df = spark.createDataFrame(
    new_data,
    columns
)

combined_df = bank_df.unionByName(new_df)

combined_df.show()

+--------------+-----------+------------+----------------+--------+---------+----------------+
|transaction_id|customer_id|account_type|transaction_type|  amount|     city|transaction_date|
+--------------+-----------+------------+----------------+--------+---------+----------------+
|             1|        101|     SAVINGS|          CREDIT| 75000.0|Bengaluru|      2026-09-01|
|             2|        102|     CURRENT|           DEBIT| 45000.0|Hyderabad|      2026-09-02|
|             3|        103|     SAVINGS|          CREDIT|120000.0|  Chennai|      2026-09-03|
|             4|        104|     CURRENT|           DEBIT| 65000.0|     Pune|      2026-09-04|
|             5|        105|     SAVINGS|           DEBIT| 30000.0|   Mumbai|      2026-09-05|
|             6|        101|     CURRENT|          CREDIT|150000.0|Bengaluru|      2026-09-06|
|             7|        106|     SAVINGS|           DEBIT| 55000.0|    Delhi|      2026-09-07|
|             8|        107|     CURRENT|         

In [15]:
# Question 15
customer_data = [
    (101, "Rahul", "Bengaluru"),
    (102, "Priya", "Hyderabad"),
    (103, "Arun", "Chennai"),
    (104, "Sneha", "Pune"),
    (105, "Kiran", "Mumbai"),
    (106, "Anil", "Delhi"),
    (107, "Meena", "Pune"),
    (108, "Ravi", "Chennai"),
    (109, "Suresh", "Hyderabad")
]

customer_columns = [
    "customer_id",
    "customer_name",
    "customer_city"
]

customer_df = spark.createDataFrame(
    customer_data,
    customer_columns
)

In [16]:
# Question 16
repartitioned_df = bank_df.repartition(
    "account_type"
)

print(
    "Number of partitions:",
    repartitioned_df.rdd.getNumPartitions()
)

Number of partitions: 1
